In [8]:
import sqlite3
import pandas as pd

DB_PATH = "../DB/oedb_baseline_v3.db"
service_account_file = "../config/service_account_key.json"

from oral_notes.s1_extract.doc_loader import GoogleDriveLoader
from oral_notes.s1_extract.text_extractor import TextExtractor

file_loader = GoogleDriveLoader(service_account_file)
extractor = TextExtractor()

rows = []

with sqlite3.connect(DB_PATH) as conn:
    cursor = conn.cursor()

    for notegroup_id in range(1, 24):
        cursor.execute("""
            SELECT project_name, phase, note_url_QA, note_url_PARTICIPANT
            FROM notegroups
            WHERE notegroupID = ?
        """, (notegroup_id,))
        row = cursor.fetchone()

        if row is None:
            print(f"[{notegroup_id}] No notegroup found — skipping")
            continue

        project_name, phase, note_url_qa, note_url_participant = row
        all_drive_paths = {}

        for label, url in [("QA", note_url_qa), ("PARTICIPANT", note_url_participant)]:
            if not url:
                continue
            try:
                result = file_loader.load(url)
                all_drive_paths[label] = result['drive_path']
            except Exception as e:
                print(f"[{notegroup_id}] {label} load failed: {e}")

        combined_drive_paths = "|".join(all_drive_paths.values())
        rows.append({"notegroupID": notegroup_id, "combined_drive_paths": combined_drive_paths})

df = pd.DataFrame(rows)

Loading: Rama - Note-taking 3 (1).docx (application/vnd.openxmlformats-officedocument.wordprocessingml.document)
Drive path: 01 extern/01 huidige klanten : projecten/1306 - M&E Helmond de Peel Regio/4. Expertpool _ klankbordgroep /Notities en analyse /Rama - Note-taking 3 (1).docx
Loading: Aanwezig sessie 1 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/1306 - M&E Helmond de Peel Regio/4. Expertpool _ klankbordgroep /Notities en analyse /Aanwezig sessie 1
Loading: Iyad - Note-taking 3.12 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/1306 - M&E Helmond de Peel Regio/4. Expertpool _ klankbordgroep /Notities en analyse /Iyad - Note-taking 3.12
Loading: Aanwezig sessie 1 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/1306 - M&E Helmond de Peel Regio/4. Expertpool _ klankbordgroep /Notities en analyse /Aanwezig sessie 1
Loading: Note form Danna 22 Nov (appl

In [9]:
pd.set_option("display.max_colwidth", None)
df

,notegroupID,combined_drive_paths
0,1,01 extern/01 huidige klanten : projecten/1306 - M&E Helmond de Peel Regio/4. Expertpool _ klankbordgroep /Notities en analyse /Rama - Note-taking 3 (1).docx|01 extern/01 huidige klanten : projecten/1306 - M&E Helmond de Peel Regio/4. Expertpool _ klankbordgroep /Notities en analyse /Aanwezig sessie 1
1,2,01 extern/01 huidige klanten : projecten/1306 - M&E Helmond de Peel Regio/4. Expertpool _ klankbordgroep /Notities en analyse /Iyad - Note-taking 3.12|01 extern/01 huidige klanten : projecten/1306 - M&E Helmond de Peel Regio/4. Expertpool _ klankbordgroep /Notities en analyse /Aanwezig sessie 1
2,3,01 extern/01 huidige klanten : projecten/ ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 1/Notes/Note form Danna 22 Nov|01 extern/01 huidige klanten : projecten/ ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 1/Deelnemers lijst + indeling
3,4,01 extern/01 huidige klanten : projecten/ ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 1/Notes/Fatih notes form 22 nov|01 extern/01 huidige klanten : projecten/ ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 1/Deelnemers lijst + indeling
4,5,01 extern/01 huidige klanten : projecten/ ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 1/Notes/Naya notes
5,6,01 extern/01 huidige klanten : projecten/ ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Notes/Copy of Ale_ Note-taking form 28.11 (English translation)|01 extern/01 huidige klanten : projecten/ ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Deelnemers lijst + indeling #Session2
6,7,01 extern/01 huidige klanten : projecten/ ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Notes/Danna: Note-taking form 28.11|01 extern/01 huidige klanten : projecten/ ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Deelnemers lijst + indeling #Session2
7,8,01 extern/01 huidige klanten : projecten/ ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Notes/Fatih: Note-taking form 28.11|01 extern/01 huidige klanten : projecten/ ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Deelnemers lijst + indeling #Session2
8,9,01 extern/01 huidige klanten : projecten/ ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Notes/Nesrine: Note-taking form 28.11|01 extern/01 huidige klanten : projecten/ ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Deelnemers lijst + indeling #Session2
9,10,01 extern/01 huidige klanten : projecten/ ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Notes/Reza: Note-taking form 29.11|01 extern/01 huidige klanten : projecten/ ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Deelnemers lijst + indeling #Session2
